# 07. Sorting in Python: Timsort, Keys and Comparators

In real code and in most interviews you will call `sort`. Knowing **how it works and how to control it** is what matters.

**Learning order:** TIMSORT → STABILITY IN PRACTICE → MULTI-KEY SORTING → KEY FUNCTIONS → CMP_TO_KEY → PITFALLS → INTERVIEW THINKING

## 1. Timsort

Python's `list.sort()` and `sorted()` use **Timsort** (a hybrid of merge sort and insertion sort):

- Finds **runs** (already sorted or reverse-sorted stretches) and merges them.
- Small chunks are sorted by insertion sort.
- **Best case O(n)** (already sorted), **worst case O(n log n)**, **stable**, O(n) extra space.

Real data is often partly sorted (logs by time, appended records), so Timsort is faster than a plain quicksort on it.

In [31]:
import random, timeit

n = 200000
sorted_data = list(range(n))
random_data = sorted_data[:]
random.shuffle(random_data)

t_sorted = timeit.timeit(lambda: sorted(sorted_data), number=3)
t_random = timeit.timeit(lambda: sorted(random_data), number=3)
print(t_sorted < t_random)   # True  (already sorted input is much faster)

True


## 2. Stability gives you multi-key sorting for free

Because the sort is **stable**, to sort by `department` then (within each department) by `salary` descending, sort by the **secondary** key first, then the **primary** key. Equal primary keys keep the earlier order.

Or do it in one step with a **tuple key**: `key=lambda p: (p.dept, -p.salary)` (negate numbers for descending).

In [32]:
people = [("Amit", "IT", 50), ("Bela", "HR", 60), ("Chen", "IT", 70), ("Dev", "HR", 60)]

two_pass = sorted(people, key=lambda p: -p[2])               # secondary: salary descending
two_pass = sorted(two_pass, key=lambda p: p[1])              # primary: department (stable)
one_pass = sorted(people, key=lambda p: (p[1], -p[2]))       # same result in one pass
print(two_pass == one_pass, one_pass)
# True [('Bela', 'HR', 60), ('Dev', 'HR', 60), ('Chen', 'IT', 70), ('Amit', 'IT', 50)]

True [('Bela', 'HR', 60), ('Dev', 'HR', 60), ('Chen', 'IT', 70), ('Amit', 'IT', 50)]


## 3. Handy key patterns

| Goal | Key |
|---|---|
| by length, then alphabetical | `key=lambda w: (len(w), w)` |
| by a dict's value, ties by key | `sorted(d.items(), key=lambda kv: (-kv[1], kv[0]))` |
| case-insensitive | `key=str.casefold` |
| by a field of a tuple/object | `operator.itemgetter(1)`, `operator.attrgetter("age")` |
| by custom order of values | `order = {v: i for i, v in enumerate(priority)}`, `key=order.get` |
| descending | `reverse=True` (still stable) or negate numbers |
| sort indexes by value | `sorted(range(n), key=lambda i: nums[i])` |

In [33]:
from operator import itemgetter

words = ["banana", "kiwi", "Apple", "fig", "cherry"]
print(sorted(words, key=lambda w: (len(w), w)))      # ['fig', 'kiwi', 'Apple', 'banana', 'cherry']
print(sorted(words, key=str.casefold))               # ['Apple', 'banana', 'cherry', 'fig', 'kiwi']

d = {"b": 2, "a": 2, "c": 5}
print(sorted(d.items(), key=lambda kv: (-kv[1], kv[0])))   # [('c', 5), ('a', 2), ('b', 2)]
print(sorted([(1, "z"), (3, "a"), (2, "m")], key=itemgetter(1)))   # [(3, 'a'), (2, 'm'), (1, 'z')]

nums = [30, 10, 20]
print(sorted(range(3), key=lambda i: nums[i]))   # [1, 2, 0]   (indexes in value order)

priority = ["high", "medium", "low"]
rank = {p: i for i, p in enumerate(priority)}
print(sorted(["low", "high", "low", "medium"], key=rank.get))   # ['high', 'medium', 'low', 'low']

['fig', 'kiwi', 'Apple', 'banana', 'cherry']
['Apple', 'banana', 'cherry', 'fig', 'kiwi']
[('c', 5), ('a', 2), ('b', 2)]
[(3, 'a'), (2, 'm'), (1, 'z')]
[1, 2, 0]
['high', 'medium', 'low', 'low']


## 4. `cmp_to_key` for pairwise rules

When the order depends on **two items together** (not a single key), write a comparator returning negative / 0 / positive and wrap it with `functools.cmp_to_key`. Example: arrange numbers to form the **smallest** number: `a` before `b` if `a + b < b + a` as strings.

In [34]:
from functools import cmp_to_key

def smallest_number(nums):
    strs = list(map(str, nums))
    strs.sort(key=cmp_to_key(lambda a, b: -1 if a + b < b + a else (1 if a + b > b + a else 0)))
    return "".join(strs)

print(smallest_number([3, 30, 34, 5, 9]))   # 3033459

3033459


## 5. Pitfalls

- `a = a.sort()` makes `a` `None`.
- Sorting **mixed types** raises `TypeError` in Python 3 (`[3, "a"]`).
- `None` cannot be compared with numbers.
- `NaN` breaks ordering (comparisons are always false).
- `reverse=True` keeps equal items in **original** order (not reversed).
- `sort` mutates the list; `sorted` returns a new list and accepts any iterable.
- Sorting costs O(n log n). Do not sort inside a loop unless needed.
- `heapq.nsmallest(k, data)` beats a full sort when `k` is small.

In [35]:
try:
    sorted([3, "a"])
except TypeError:
    print("TypeError")   # TypeError

x = [2, 1]
print(x.sort())   # None

TypeError
None


## 6. Interview thinking

**Say:** "I'll use the built-in sort, which is Timsort: O(n log n) worst case, O(n) on sorted input, stable. For the tie-break I put it in a tuple key."

**When to write your own sort:** only when the problem is *about* sorting (merge sort, quicksort, counting sort) or says "do not use the built-in sort".

**Common mistakes:** descending order with `reverse=True` when a stable tie-break was needed (use negated keys), comparator functions without `cmp_to_key`, assuming the sort is in place (`sorted` is not).

## Quick Revision

- Timsort: O(n log n), best O(n), stable, O(n) space.
- Multi-key: tuple key with negated numbers, or sort by secondary then primary (stability).
- `cmp_to_key` for pairwise comparison rules.
- `sort()` returns `None`. Mixed types raise `TypeError`.